In [ ]:
# ===============================================================
# Lab 3 (ML) - Фінальна, Оптимізована та Безпомилкова Версія (Виправлення NLTK)
# ===============================================================

# --- 1. Імпорт бібліотек та відключення попереджень ---
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import gensim
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler 
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
import seaborn as sns
import matplotlib.pyplot as plt
import warnings

# Відключення всіх попереджень
warnings.filterwarnings('ignore') 

# --- УНІВЕРСАЛЬНЕ ЗАВАНТАЖЕННЯ NLTK РЕСУРСІВ (ВИПРАВЛЕНО) ---
# Використовуємо простий nltk.download(), щоб уникнути проблем із LookupError та DownloadError.
required_nltk_data = ['stopwords', 'punkt', 'wordnet']
for resource in required_nltk_data:
    try:
        # Спроба знайти ресурс. Якщо його немає, nltk.data.find викличе виняток.
        nltk.data.find(f'corpora/{resource}') 
    except:
        # Якщо ресурс не знайдено, завантажуємо його.
        print(f"Завантажую ресурс NLTK: {resource}")
        nltk.download(resource, quiet=True) 
print("Всі необхідні ресурси NLTK завантажено.")
# -------------------------------------------------------------------


# --- 2. Завантаження та попередня обробка даних ---
# РЕКОМЕНДАЦІЯ: для "гарних результатів" використовуйте Corona_NLP_train.csv
df = pd.read_csv('Corona_NLP_test.csv', encoding='latin-1') 
df = df[['OriginalTweet', 'Sentiment']]
df = df.dropna()

# Конвертуємо 5 класів у 3 (Negative, Positive, Other)
def simplify_sentiment(sentiment):
    if 'Negative' in sentiment:
        return 'Negative'
    elif 'Positive' in sentiment:
        return 'Positive'
    else: 
        return 'Other'

df['Sentiment'] = df['Sentiment'].apply(simplify_sentiment)

print("\nРозподіл класів (3 категорії):")
print(df['Sentiment'].value_counts())

# Лематизатор та стоп-слова
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|https\S+", '', text)  
    text = re.sub(r"[^a-z\s]", '', text)  
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in stop_words and len(w) > 1]
    tokens = [lemmatizer.lemmatize(w) for w in tokens] 
    return tokens

df['tokens'] = df['OriginalTweet'].apply(preprocess_text)


# --- 3. Кастомний Word2Vec Transformer ---
class Word2VecVectorizer(BaseEstimator, TransformerMixin):
    def __init__(self, vector_size=200, window=5, min_count=2, workers=4):
        self.vector_size = vector_size
        self.window = window
        self.min_count = min_count
        self.workers = workers
        self.model = None

    def fit(self, X, y=None):
        self.model = Word2Vec(X, vector_size=self.vector_size,
                              window=self.window, min_count=self.min_count,
                              workers=self.workers)
        return self

    def transform(self, X):
        vectors = []
        for tokens in X:
            vec = np.zeros(self.vector_size)
            count = 0
            for word in tokens:
                if word in self.model.wv:
                    vec += self.model.wv[word]
                    count += 1
            if count != 0:
                vec /= count
            vectors.append(vec)
        vectors_np = np.vstack(vectors)
        return vectors_np


# --- 4. Поділ на train/test ---
X = df['tokens'] 
y = df['Sentiment']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


# --- 5. Pipeline та порівняння моделей ---
pca_components = [None, 50, 100, 200] 
results = []
best_accuracy = 0
best_pipeline = None
best_params = {}

classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=1000, solver='lbfgs', random_state=42), 
    "SVM": SVC(kernel='linear', C=1.0, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42),
    "Naive Bayes": GaussianNB() 
}

for name, classifier in classifiers.items():
    print(f"\n--- Модель: {name} ---")
    for n_comp in pca_components:
        
        steps = [('word2vec', Word2VecVectorizer(vector_size=200))]
        
        # Додаємо стандартизацію для LR, SVM, RF
        if name != "Naive Bayes":
            steps.append(('scaler', StandardScaler())) 
        
        if n_comp is not None:
            steps.append(('pca', PCA(n_components=n_comp)))
        
        steps.append(('classifier', classifier))
        pipeline = Pipeline(steps)
        
        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)
        
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='weighted')
        
        results.append({
            'Model': name,
            'PCA Components': 'No PCA' if n_comp is None else n_comp,
            'Accuracy': acc,
            'F1-score': f1
        })

        if acc > best_accuracy:
            best_accuracy = acc
            best_pipeline = pipeline
            best_params = {'Model': name, 'PCA Components': 'No PCA' if n_comp is None else n_comp}

        print(f"  PCA={n_comp if n_comp is not None else 'No PCA'}: Acc={acc:.4f}, F1={f1:.4f}")

# --- 6. Тюнінг гіперпараметрів для найкращої моделі (Random Forest) ---
print("\n--- 6. Тюнінг гіперпараметрів (GridSearchCV) для Random Forest (PCA=100) ---")

w2v_pca_rf_pipeline = Pipeline([
    ('word2vec', Word2VecVectorizer(vector_size=200)),
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=100)), 
    ('classifier', RandomForestClassifier(random_state=42))
])

param_grid = {
    'classifier__n_estimators': [200, 300, 400], 
    'classifier__max_depth': [None, 10, 20],      
    'classifier__min_samples_split': [2, 5]     
}

grid_search = GridSearchCV(w2v_pca_rf_pipeline, param_grid, cv=3, scoring='accuracy', verbose=0)
grid_search.fit(X_train, y_train)

y_pred_tuned = grid_search.predict(X_test)
tuned_acc = accuracy_score(y_test, y_pred_tuned)
tuned_f1 = f1_score(y_test, y_pred_tuned, average='weighted')

print(f"НАЙКРАЩІ ПАРАМЕТРИ: {grid_search.best_params_}")
print(f"Точність після тюнінгу: {tuned_acc:.4f}")
print(f"F1-score після тюнінгу: {tuned_f1:.4f}")

if tuned_acc > best_accuracy:
    best_accuracy = tuned_acc
    best_pipeline = grid_search.best_estimator_
    best_params = {'Model': 'Random Forest (Tuned)', 'PCA Components': 100}

# --- 7. Фінальна таблиця результатів та візуалізація ---
results_df = pd.DataFrame(results)

results_df.loc[len(results_df)] = {
    'Model': 'Random Forest (Tuned)', 
    'PCA Components': 100, 
    'Accuracy': tuned_acc, 
    'F1-score': tuned_f1
}

# ВИПРАВЛЕННЯ ПОМИЛКИ TypeError: Конвертуємо всі значення PCA Components до рядкового типу
results_df['PCA Components'] = results_df['PCA Components'].astype(str) 

print("\nТАБЛИЦЯ ФІНАЛЬНИХ РЕЗУЛЬТАТІВ:")
display(results_df.sort_values(by='Accuracy', ascending=False).head(5))

# Візуалізація впливу PCA
plt.figure(figsize=(10,6))
sns.lineplot(data=results_df[results_df['Model'].str.contains('Tuned') == False], 
             x='PCA Components', y='Accuracy', hue='Model', marker='o')
plt.title("Вплив PCA на точність класифікації")
plt.show()

# --- 8. Оцінка найкращої моделі ---
print("\n--- 8. Оцінка Фінальної Найкращої Моделі ---")
y_pred_final = best_pipeline.predict(X_test)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_final)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=np.unique(y_test), yticklabels=np.unique(y_test))
plt.title(f'Confusion Matrix ({best_params["Model"]}, PCA={best_params["PCA Components"]})')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

# Classification Report
print("Classification Report для найкращої моделі:")
print(classification_report(y_test, y_pred_final))

Завантажую ресурс NLTK: punkt
Завантажую ресурс NLTK: wordnet
Всі необхідні ресурси NLTK завантажено.

Розподіл класів (3 категорії):
Sentiment
Negative    1633
Positive    1546
Other        619
Name: count, dtype: int64

--- Модель: Logistic Regression ---
  PCA=No PCA: Acc=0.5737, F1=0.5591
  PCA=50: Acc=0.5408, F1=0.5254
  PCA=100: Acc=0.5553, F1=0.5402
  PCA=200: Acc=0.5697, F1=0.5544

--- Модель: SVM ---
  PCA=No PCA: Acc=0.5579, F1=0.5366
  PCA=50: Acc=0.5105, F1=0.4856
  PCA=100: Acc=0.5474, F1=0.5208
  PCA=200: Acc=0.5618, F1=0.5397

--- Модель: Random Forest ---
  PCA=No PCA: Acc=0.4684, F1=0.4583
  PCA=50: Acc=0.5513, F1=0.5351
  PCA=100: Acc=0.5368, F1=0.5185
  PCA=200: Acc=0.5553, F1=0.5244

--- Модель: Naive Bayes ---
  PCA=No PCA: Acc=0.4184, F1=0.4164
  PCA=50: Acc=0.5329, F1=0.5328
  PCA=100: Acc=0.5579, F1=0.5595
  PCA=200: Acc=0.5289, F1=0.5363

--- 6. Тюнінг гіперпараметрів (GridSearchCV) для Random Forest (PCA=100) ---
